### Pipeline Parallelism Experiments in PyTorch

## Overview
This notebook demonstrates three progressive experiments with pipeline parallelism using PyTorch's distributed.pipelining module. 
Each experiment shows different aspects of model parallelism and distributed training.

#### Experiment 1: Basic Pipeline (2 Stages)
- Simple model with 2 stages:
  - Stage 0: Linear(4→3) + ReLU
  - Stage 1: Linear(3→1)
- Demonstrates basic pipeline setup with:
  - Microbatching (batch_size=8, chunks=4)
  - Forward pass only (model.eval())
  - 2 worker processes
- Focus: Understanding basic pipeline mechanics

#### Experiment 2: Training Pipeline (2 Stages)
- Deeper model architecture:
  - Stage 0: Linear(4→16) → ReLU → Linear(16→8) → ReLU
  - Stage 1: Linear(8→4) → ReLU → Linear(4→1)
- Introduces training components:
  - Synthetic regression dataset (X³ + X² + 1)
  - MSE loss and Adam optimizer
  - 1000 training epochs
  - Larger batch size (64) with 4 chunks
- Focus: Pipeline parallel training workflow

#### Experiment 3: Multi-Stage Pipeline (10 Stages)
- Deep network with 10 layers:
  - Progressive dimension reduction: 4→32→32→32→32→16→16→8→8→4→1
  - Automatic stage splitting based on module count
- Advanced features:
  - Dynamic chunk size adjustment
  - Per-stage parameter management
  - Flexible optimizer creation
  - 10 worker processes
  - Large batch training (batch_size=5000)
  - Extended training (5000 epochs)
- Focus: Scaling pipeline parallelism to many stages

#### Key Concepts Demonstrated
- Pipeline stage creation and management
- Microbatch handling and scheduling
- Distributed training coordination
- Memory efficient training with pipeline parallelism
- Forward/backward pass orchestration


In [ ]:
%%writefile model.py

import os
import argparse
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.distributed.pipelining import pipeline, ScheduleGPipe, SplitPoint

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Stage 0: Linear(4→3) + ReLU
        self.stage0 = nn.Sequential(
            nn.Linear(4, 3),
            nn.ReLU()
        )
        # Stage 1: Linear(3→1)
        self.stage1 = nn.Sequential(
            nn.Linear(3, 1)
        )

    def forward(self, x):
        x = self.stage0(x)
        x = self.stage1(x)
        return x


def run(args):
    torch.manual_seed(0)
    device = args.device

    model = MyModel().to(device)
    model.eval()

    # --- Microbatch setup ---
    batch_size = args.batch_size           
    chunks = args.chunks                   
    assert batch_size % chunks == 0, "batch_size must be divisible by chunks"
    microbatch_size = batch_size // chunks

    # Full batch for actual forward (schedule will split automatically)
    full_batch = torch.randn(batch_size, 4, device=device)

    # Single microbatch for pipeline tracing
    mb_one = torch.randn(microbatch_size, 4, device=device)

    # --- Split between stage0 and stage1 ---
    split_spec = {'stage1': SplitPoint.BEGINNING}

    # --- Build pipeline graph ---
    pipe = pipeline(
        model,
        mb_args=(mb_one,),   # must reflect one microbatch shape
        split_spec=split_spec,
    )

    assert pipe.num_stages == args.world_size, (
        f"Pipeline stages ({pipe.num_stages}) != world_size ({args.world_size})"
    )

    stage_module = pipe.get_stage_module(args.rank)
    print(f"\n[Rank {args.rank}] ---------- Readable FX Graph ----------")
    stage_module.print_readable()

    print(f"[Rank {args.rank}] ---------- Node summary ----------")


    # --- Build runtime stage and GPipe schedule ---
    stage = pipe.build_stage(args.rank, device=device)
    schedule = ScheduleGPipe(stage, chunks)
    
    # --- Execute one forward pass ---
    if args.rank == 0:
        out = schedule.step(full_batch)
        if out is not None:
            print(f"[Rank 0] Output shape: {out.shape}")
    else:
        _ = schedule.step()

    dist.barrier()
    dist.destroy_process_group()
    print(f"[Rank {args.rank}] complete.")

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--world_size", type=int,
                        default=int(os.getenv("WORLD_SIZE", 2)))
    parser.add_argument("--rank", type=int,
                        default=int(os.getenv("RANK", -1)))
    parser.add_argument("--chunks", type=int, default=4)     # micro-batches
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--cuda", action="store_true")
    args = parser.parse_args()

    if args.cuda:
        dev_id = args.rank % torch.cuda.device_count()
        args.device = torch.device(f"cuda:{dev_id}")
        backend = "nccl"
    else:
        args.device = torch.device("cpu")
        backend = "gloo"

    dist.init_process_group(backend=backend,
                            rank=args.rank,
                            world_size=args.world_size)

    run(args)


if __name__ == "__main__":
    main()



Overwriting model.py


In [54]:
!export OMP_NUM_THREADS=5
!torchrun --nproc_per_node=2 model.py


W1015 13:48:21.874000 90944 torch/distributed/run.py:766] 
W1015 13:48:21.874000 90944 torch/distributed/run.py:766] *****************************************
W1015 13:48:21.874000 90944 torch/distributed/run.py:766] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1015 13:48:21.874000 90944 torch/distributed/run.py:766] *****************************************

[Rank 1] ---------- Readable FX Graph ----------
class GraphModule(torch.nn.Module):
    def forward(self, relu: "f32[2, 3]"):
        # No stacktrace found for following nodes
        stage1: "f32[2, 1]" = self.stage1(relu);  relu = None
        return stage1
        
    class stage1(torch.nn.Module):
        def forward(self, relu: "f32[2, 3]"):
            # No stacktrace found for following nodes
            _0: "f32[2, 1]" = getattr(self, "0")(relu);  relu = Non

In [ ]:
%%writefile model.py
import os
import argparse
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.distributed.pipelining import pipeline, ScheduleGPipe, SplitPoint

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.stage0 = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
        )
        self.stage1 = nn.Sequential(
            nn.Linear(8, 4),
            nn.ReLU(),
            nn.Linear(4, 1),
        )

    def forward(self, x):
        x = self.stage0(x)
        x = self.stage1(x)
        return x

def run(args):
    torch.manual_seed(0)
    device = args.device

    model = MyModel().to(device)
    model.train()

    batch_size = args.batch_size
    chunks = args.chunks
    microbatch_size = batch_size // chunks

    # ----- contiguous synthetic dataset -----
    X = torch.stack([torch.linspace(-2, 2, batch_size) for _ in range(4)], dim=1)
    X = X.contiguous().to(device)                      # ensure row-major (C) layout
    Y = (X ** 3 + X ** 2 + 1).sum(dim=1, keepdim=True)

    # ----- microbatch sample (same layout) -----
    mb_one = X[:microbatch_size].clone().contiguous()

    print(
        f"[Rank {args.rank}] X.stride={X.stride()}, "
        f"mb_one.stride={mb_one.stride()}, "
        f"contiguous={X.is_contiguous()}"
    )

    split_spec = {"stage1": SplitPoint.BEGINNING}

    # Build pipeline graph + per-rank stage
    pipe = pipeline(model, mb_args=(mb_one,), split_spec=split_spec)
    stage_module = pipe.get_stage_module(args.rank)
    stage_module.train()
    stage = pipe.build_stage(args.rank, device=device)
    schedule = ScheduleGPipe(stage, chunks)

    optimizer = torch.optim.Adam(stage_module.parameters(), lr=args.lr)
    loss_fn = nn.MSELoss()

    for epoch in range(args.epochs):
        optimizer.zero_grad()

        if args.rank == 0:
            # Rank 0: drive the pipeline forward
            _ = schedule.step(X)
        else:
            # Rank 1: receive activations, compute loss, backward, optimize
            output = schedule.step()
            loss = loss_fn(output, Y)
            loss.backward()
            optimizer.step()
            if epoch % 100 == 0:
                print(f"[Rank 1] Epoch {epoch:03d} | Loss = {loss.item():.6f}")

        dist.barrier()

    dist.destroy_process_group()
    print(f"[Rank {args.rank}] training complete.")

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--world_size", type=int, default=int(os.getenv("WORLD_SIZE", 2)))
    parser.add_argument("--rank", type=int, default=int(os.getenv("RANK", -1)))
    parser.add_argument("--chunks", type=int, default=4)
    parser.add_argument("--batch_size", type=int, default=64)
    parser.add_argument("--epochs", type=int, default=1000)
    parser.add_argument("--lr", type=float, default=1e-2)
    parser.add_argument("--cuda", action="store_true")
    args = parser.parse_args()

    if args.cuda:
        dev_id = args.rank % torch.cuda.device_count()
        args.device = torch.device(f"cuda:{dev_id}")
        backend = "nccl"
    else:
        args.device = torch.device("cpu")
        backend = "gloo"

    dist.init_process_group(backend=backend, rank=args.rank, world_size=args.world_size)
    run(args)


if __name__ == "__main__":
    main()



Overwriting model.py


In [80]:
!export OMP_NUM_THREADS=5
!torchrun --nproc_per_node=2 model.py

W1015 15:20:17.802000 91937 torch/distributed/run.py:766] 
W1015 15:20:17.802000 91937 torch/distributed/run.py:766] *****************************************
W1015 15:20:17.802000 91937 torch/distributed/run.py:766] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1015 15:20:17.802000 91937 torch/distributed/run.py:766] *****************************************
[Rank 0] X.stride=(4, 1), mb_one.stride=(4, 1), contiguous=True
[Rank 1] X.stride=(4, 1), mb_one.stride=(4, 1), contiguous=True
[Rank 1] Epoch 000 | Loss = 284.910736
[Rank 1] Epoch 100 | Loss = 208.250610
[Rank 1] Epoch 200 | Loss = 93.128052
[Rank 1] Epoch 300 | Loss = 35.061386
[Rank 1] Epoch 400 | Loss = 19.712307
[Rank 1] Epoch 500 | Loss = 14.508719
[Rank 1] Epoch 600 | Loss = 10.996244
[Rank 1] Epoch 700 | Loss = 8.103483
[Rank 1] Epoch 800 | Loss = 5.851472
[Ra

In [ ]:
%%writefile model.py

import os
import argparse
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.distributed.pipelining import pipeline, ScheduleGPipe, SplitPoint


class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        dims = [4, 32, 32, 32, 32, 16, 16, 8, 8, 4, 1]
        layers = []
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            if i < len(dims) - 2:
                layers.append(nn.ReLU())
        # make them named children under a container so split_spec can refer to net.<idx>
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def run(args):
    torch.manual_seed(0)
    device = args.device

    model = MyModel().to(device)
    model.train()

    batch_size = args.batch_size
    num_stages = args.world_size

    # --- ensure chunks valid ---
    requested_chunks = max(args.chunks, num_stages)
    chunks = requested_chunks
    while chunks <= batch_size and (batch_size % chunks) != 0:
        chunks += 1
    if chunks > batch_size:
        raise ValueError(
            f"Cannot find chunks >= stages ({num_stages}) dividing batch_size ({batch_size}). "
            "Increase batch_size or reduce world_size."
        )
    if chunks != args.chunks:
        if args.rank == 0:
            print(f"[Rank {args.rank}] Adjusted chunks: requested={args.chunks} -> using {chunks} "
                  f"(world_size={num_stages}, batch_size={batch_size})")

    microbatch_size = batch_size // chunks

    # --- create contiguous synthetic dataset ---
    X = torch.stack([torch.linspace(-2, 2, batch_size) for _ in range(4)], dim=1)
    X = X.contiguous().to(device)
    Y = (X ** 3 + X ** 2 + 1).sum(dim=1, keepdim=True)

    # microbatch sample derived from X (same layout)
    mb_one = X[:microbatch_size].clone().contiguous()

    # diagnostic print
    print(
        f"[Rank {args.rank}] X.stride={X.stride()}, mb_one.stride={mb_one.stride()}, "
        f"contiguous={X.is_contiguous()}, chunks={chunks}"
    )

    # --- split_spec: split the 'net' sequential into roughly equal contiguous chunks ---
    # len(model.net) is number of modules (linear + relu entries)
    modules_total = len(model.net)
    # compute roughly equal splits by modules count
    step = max(1, modules_total // num_stages)
    split_spec = {
        f"net.{i * step}": SplitPoint.BEGINNING
        for i in range(1, num_stages)
    }

    # --- build pipeline and per-rank stage module ---
    pipe = pipeline(model, mb_args=(mb_one,), split_spec=split_spec)
    stage_module = pipe.get_stage_module(args.rank)   # GraphModule with parameters (maybe none)
    stage_module.train()

    # runtime stage & schedule
    stage = pipe.build_stage(args.rank, device=device)
    schedule = ScheduleGPipe(stage, chunks)

    # --- create optimizer only if this stage has params ---
    # materialize list to examine
    params = list(stage_module.parameters())
    param_count = sum(p.numel() for p in params)
    if param_count > 0:
        optimizer = torch.optim.Adam(params, lr=args.lr)
        if args.rank == 0:
            print(f"[Rank {args.rank}] Created optimizer with {param_count} params")
        else:
            print(f"[Rank {args.rank}] Created optimizer with {param_count} params")
    else:
        optimizer = None
        print(f"[Rank {args.rank}] No trainable parameters in this stage (optimizer skipped)")

    loss_fn = nn.MSELoss()

    # --- training loop ---
    for epoch in range(args.epochs):
        # zero grad only on ranks that have optimizer (and therefore parameters)
        if optimizer is not None:
            optimizer.zero_grad()

        if args.rank == 0:
            # rank 0 drives the pipeline forward with full batch
            _ = schedule.step(X)
        elif args.rank == args.world_size - 1:
            # last rank: receives activations, computes loss and backward, then optimizer.step()
            output = schedule.step()
            loss = loss_fn(output, Y)
            # compute gradients across the pipeline
            loss.backward()
            if optimizer is not None:
                optimizer.step()
            if epoch % 1000 == 0:
                print(f"[Rank {args.rank}] Epoch {epoch:03d} | Loss = {loss.item():.6f}")
        else:
            # intermediate ranks: participate in pipeline; after schedule returns the grads are ready
            _ = schedule.step()
            # update local params (if any)
            if optimizer is not None:
                optimizer.step()

        # synchronize for neat logging (optional)
        dist.barrier()

    # clean up
    dist.destroy_process_group()
    print(f"[Rank {args.rank}] training complete.")

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--world_size", type=int, default=int(os.getenv("WORLD_SIZE", 10)))
    parser.add_argument("--rank", type=int, default=int(os.getenv("RANK", -1)))
    parser.add_argument("--chunks", type=int, default=10)
    parser.add_argument("--batch_size", type=int, default=5000)
    parser.add_argument("--epochs", type=int, default=5000)
    parser.add_argument("--lr", type=float, default=5e-2)
    parser.add_argument("--cuda", action="store_true")
    args = parser.parse_args()

    if args.cuda:
        dev_id = args.rank % torch.cuda.device_count()
        args.device = torch.device(f"cuda:{dev_id}")
        backend = "nccl"
    else:
        args.device = torch.device("cpu")
        backend = "gloo"

    dist.init_process_group(backend=backend, rank=args.rank, world_size=args.world_size)
    run(args)


if __name__ == "__main__":
    main()



Overwriting model.py


In [14]:
!export OMP_NUM_THREADS=1
!torchrun --nproc-per-node=10 model.py --batch_size 5000 --chunks 10


W1015 15:51:06.342000 94296 torch/distributed/run.py:766] 
W1015 15:51:06.342000 94296 torch/distributed/run.py:766] *****************************************
W1015 15:51:06.342000 94296 torch/distributed/run.py:766] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1015 15:51:06.342000 94296 torch/distributed/run.py:766] *****************************************
[Rank 8] X.stride=(4, 1), mb_one.stride=(4, 1), contiguous=True, chunks=10
[Rank 0] X.stride=(4, 1), mb_one.stride=(4, 1), contiguous=True, chunks=10
[Rank 7] X.stride=(4, 1), mb_one.stride=(4, 1), contiguous=True, chunks=10
[Rank 4] X.stride=(4, 1), mb_one.stride=(4, 1), contiguous=True, chunks=10
[Rank 6] X.stride=(4, 1), mb_one.stride=(4, 1), contiguous=True, chunks=10
[Rank 9] X.stride=(4, 1), mb_one.stride=(4, 1), contiguous=True, chunks=10
[Rank 2] X.stride=(4, 1